# Sanity/Manual Value Check: API Window Comparison from `all_data.parquet`

Dieses Notebook zeigt für jede API ein **6-Stunden-Fenster** aus `data/processed/all_data.parquet`:

- **01.03.2021 ab 10:00**
- **01.08.2025 ab 10:00**

Die Ausgabe erfolgt je API mit dem **passenden Zeitstempel** (`timestamp_utc` oder `timestamp_cet`) und **allen zugehörigen Spalten**.


Hinweis: In diesem Notebook wird **Netztransparenz als UTC** abgefragt/verglichen.

In [10]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Resolve project data path from either repo root, notebooks/, or notebooks/api/.
candidates = [
    Path('data/processed/all_data.parquet'),
    Path('../data/processed/all_data.parquet'),
    Path('../../data/processed/all_data.parquet'),
]
DATA_PATH = next((p for p in candidates if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError('Could not find data/processed/all_data.parquet from current working directory')

df = pd.read_parquet(DATA_PATH)

# Keep timestamp semantics explicit: UTC stays UTC, CET/CEST stays Europe/Berlin.
if 'timestamp_utc' in df.columns:
    df['timestamp_utc'] = pd.to_datetime(df['timestamp_utc'], utc=True)
if 'timestamp_cet' in df.columns:
    df['timestamp_cet'] = pd.to_datetime(df['timestamp_cet'], errors='coerce')

if 'timestamp_utc' in df.columns:
    df = df.sort_values('timestamp_utc')

print(f'Rows: {len(df):,}, Columns: {len(df.columns)}')
print('Loaded:', DATA_PATH)


# Show all columns in notebook tables
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)


Rows: 44,568, Columns: 98
Loaded: ../../data/processed/all_data.parquet


In [11]:
WINDOWS = [
    ('2021-03-01 10:00', 'Window A'),
    ('2025-08-01 10:00', 'Window B'),
]

API_CONFIG = {
    'Energy Charts (UTC)': {
        'time_col': 'timestamp_utc',
        'cols': [
            'da_price_AT','da_price_BE','da_price_CH','da_price_CZ','da_price_DK1',
            'da_price_DK2','da_price_FR','da_price_NL','da_price_PL','da_price_SE4',
        ],
    },
    'ENTSO-E (UTC)': {
        'time_col': 'timestamp_utc',
        'cols': [c for c in df.columns if c.endswith('_entsoe')] ,
    },
    'Netztransparenz (UTC)': {
        'time_col': 'timestamp_utc',
        'cols': [
            'NRV_balance_qs','NRV_balance_op','NRV_balance','reBAP_shortage_surplus',
            'rz_saldo_mw_qs','rz_saldo_mw_op','rz_saldo_mw',
            'afrr_activated_mw_pos','afrr_activated_mw_neg','mfrr_activated_mw_pos','mfrr_activated_mw_neg',
            'afrr_picasso_mw_pos','afrr_picasso_mw_neg','mfrr_mari_mw_pos','mfrr_mari_mw_neg',
        ],
    },
    'Regelleistung (CET/CEST)': {
        'time_col': 'timestamp_cet',
        'cols': [
            'afrr_capacity_offered_mw_neg','afrr_capacity_offered_mw_pos',
            'afrr_capacity_price_neg','afrr_capacity_price_pos',
            'afrr_avg_activation_price_neg','afrr_avg_activation_price_pos',
            'afrr_activation_offered_mw_neg','afrr_activation_offered_mw_pos',
            'net_import_export_mw',
            'afrr_bid_avg_activation_price_neg','afrr_bid_avg_activation_price_pos',
            'afrr_bid_vwap_activation_price_neg','afrr_bid_vwap_activation_price_pos',
            'bid_alloc_mw_neg','bid_alloc_mw_pos',
        ],
    },
    'SMARD (CET/CEST)': {
        'time_col': 'timestamp_cet',
        'cols': [
            'residual_load_actual','wind_offshore_actual','solar_actual','wind_onshore_forecast','wind_offshore_forecast',
            'solar_forecast','wind_onshore_actual',
            'generation_fossil_brown_coal_mw','generation_fossil_hard_coal_mw','generation_fossil_gas_mw',
            'generation_nuclear_mw','generation_hydro_pumped_storage_mw',
            'da_price_eur','price_intraday_eur','wind_forecast_de',
            'wind_onshore_error','wind_offshore_error','solar_error','system_stress_signal',
            'wind_onshore_capacity','wind_offshore_capacity','solar_capacity','gas_capacity',
            'hard_coal_capacity','lignite_capacity','pumped_storage_capacity',
        ],
    },
    'yfinance (UTC)': {
        'time_col': 'timestamp_utc',
        'cols': ['gas_price_ttf','coal_price_api2','co2_price_eua'],
    },
}

# Keep only existing columns (robust against schema changes)
for api, cfg in API_CONFIG.items():
    cfg['cols'] = [c for c in cfg['cols'] if c in df.columns]


In [12]:
def slice_window(data: pd.DataFrame, time_col: str, start_ts: pd.Timestamp, hours: int = 6):
    end_ts = start_ts + pd.Timedelta(hours=hours - 1)
    return data[(data[time_col] >= start_ts) & (data[time_col] <= end_ts)].copy(), end_ts


def show_api_windows(api_name: str, cfg: dict) -> None:
    time_col = cfg['time_col']
    cols = cfg['cols']

    if time_col not in df.columns:
        print(f'[{api_name}] Skipped: {time_col} not in dataset')
        return
    if not cols:
        print(f'[{api_name}] Skipped: no matching columns in dataset')
        return

    print('\n' + '=' * 120)
    print(api_name)
    print('=' * 120)

    out_cols = [time_col] + cols

    for start_txt, label in WINDOWS:
        # Interpretiere Startzeit in UTC oder CET/CEST je API.
        if time_col == 'timestamp_cet':
            start = pd.Timestamp(start_txt, tz='Europe/Berlin')
        else:
            start = pd.Timestamp(start_txt, tz='UTC')

        win, end = slice_window(df, time_col, start, hours=6)
        print(f"\n{label}: {start} -> {end} ({time_col}) | rows={len(win)}")
        display(win[out_cols].sort_values(time_col).reset_index(drop=True))


In [13]:
show_api_windows('Energy Charts (UTC)', API_CONFIG['Energy Charts (UTC)'])


Energy Charts (UTC)

Window A: 2021-03-01 10:00:00+00:00 -> 2021-03-01 15:00:00+00:00 (timestamp_utc) | rows=6


,timestamp_utc,da_price_AT,da_price_BE,da_price_CH,da_price_CZ,da_price_DK1,da_price_DK2,da_price_FR,da_price_NL,da_price_PL,da_price_SE4
0,2021-03-01 10:00:00+00:00,53.01,43.40,53.07,48.09,41.00,43.41,48.74,41.00,68.33,43.41
1,2021-03-01 11:00:00+00:00,49.88,40.04,49.29,45.90,41.04,41.04,45.02,40.04,68.44,41.04
2,2021-03-01 12:00:00+00:00,40.68,39.87,48.96,43.25,39.91,39.91,40.34,39.87,68.32,39.10
3,2021-03-01 13:00:00+00:00,40.03,40.03,50.17,43.00,40.03,40.03,40.03,40.03,67.70,39.07
4,2021-03-01 14:00:00+00:00,43.44,39.90,52.62,45.92,40.75,40.75,40.82,39.90,67.63,40.75
5,2021-03-01 15:00:00+00:00,49.03,43.33,54.41,51.10,42.00,42.08,43.84,42.00,67.71,42.08



Window B: 2025-08-01 10:00:00+00:00 -> 2025-08-01 15:00:00+00:00 (timestamp_utc) | rows=6


,timestamp_utc,da_price_AT,da_price_BE,da_price_CH,da_price_CZ,da_price_DK1,da_price_DK2,da_price_FR,da_price_NL,da_price_PL,da_price_SE4
0,2025-08-01 10:00:00+00:00,62.90,64.84,73.70,72.48,77.05,77.05,31.87,69.76,72.15,72.10
1,2025-08-01 11:00:00+00:00,65.24,51.79,64.50,63.67,61.97,61.97,25.06,55.76,76.09,60.49
2,2025-08-01 12:00:00+00:00,51.67,41.29,66.08,50.41,49.06,49.06,20.01,44.25,79.65,47.88
3,2025-08-01 13:00:00+00:00,52.30,39.59,71.82,50.76,49.10,49.10,13.43,43.21,88.15,47.92
4,2025-08-01 14:00:00+00:00,74.69,55.84,74.95,72.40,69.93,69.93,17.42,62.23,83.98,68.25
5,2025-08-01 15:00:00+00:00,77.04,56.13,80.31,77.48,83.59,83.59,27.47,76.58,90.73,81.47


- [X] All columns (see [Average day-ahead electricity spot market prices at 01.08.2025](https://www.energy-charts.info/charts/price_average_map/chart.htm?l=en&c=DE&interval=day&datetimepicker=01.08.2025))


## ENTSO-E (UTC)

In [14]:
show_api_windows('ENTSO-E (UTC)', API_CONFIG['ENTSO-E (UTC)'])


ENTSO-E (UTC)

Window A: 2021-03-01 10:00:00+00:00 -> 2021-03-01 15:00:00+00:00 (timestamp_utc) | rows=6


,timestamp_utc,load_actual_entsoe,load_forecast_da_entsoe,wind_onshore_actual_entsoe,wind_offshore_actual_entsoe,solar_actual_entsoe,biomass_actual_entsoe,hydro_reservoir_actual_entsoe,hydro_pumped_actual_entsoe,hydro_ror_actual_entsoe,wind_onshore_forecast_da_entsoe,wind_offshore_forecast_da_entsoe,solar_forecast_da_entsoe,wind_onshore_forecast_id_entsoe,wind_offshore_forecast_id_entsoe,solar_forecast_id_entsoe
0,2021-03-01 10:00:00+00:00,70779.0300,67731.6275,1774.9450,393.7850,19074.3050,4496.6950,71.4525,842.5900,1390.5200,1749.5625,430.0500,21615.3125,1099.9075,29.50,12620.3000
1,2021-03-01 11:00:00+00:00,70660.9250,67483.5800,1706.3825,358.0525,21723.5500,4437.3750,43.1075,105.0375,1397.6050,1714.2450,459.9875,23677.4900,1058.3700,34.25,13533.8650
2,2021-03-01 12:00:00+00:00,68959.0000,66297.3650,1453.5675,402.0525,22751.1425,4439.0750,33.1050,124.4850,1385.6825,1819.9325,506.2125,23374.5300,1086.1025,32.75,13605.5925
3,2021-03-01 13:00:00+00:00,67160.6750,64822.0025,1593.1775,374.1200,20763.5325,4456.9175,60.5600,47.9000,1386.4975,1964.1750,560.2400,20342.4850,1168.6350,36.25,12391.8425
4,2021-03-01 14:00:00+00:00,65874.5575,63970.9925,1783.9600,322.2075,15623.2075,4482.8875,61.8575,265.4925,1382.5075,2169.0850,613.0175,14675.8075,1353.5875,44.00,9518.1675
5,2021-03-01 15:00:00+00:00,65067.5350,63129.2575,1967.2500,353.3000,8157.5900,4580.2900,64.7325,1366.3300,1397.1300,2600.5050,645.7300,7466.2150,1662.7675,57.50,5073.7625



Window B: 2025-08-01 10:00:00+00:00 -> 2025-08-01 15:00:00+00:00 (timestamp_utc) | rows=6


,timestamp_utc,load_actual_entsoe,load_forecast_da_entsoe,wind_onshore_actual_entsoe,wind_offshore_actual_entsoe,solar_actual_entsoe,biomass_actual_entsoe,hydro_reservoir_actual_entsoe,hydro_pumped_actual_entsoe,hydro_ror_actual_entsoe,wind_onshore_forecast_da_entsoe,wind_offshore_forecast_da_entsoe,solar_forecast_da_entsoe,wind_onshore_forecast_id_entsoe,wind_offshore_forecast_id_entsoe,solar_forecast_id_entsoe
0,2025-08-01 10:00:00+00:00,56965.35900,60592.3150,8091.035261,1294.95700,29888.615857,3541.110144,100.4850,0.8900,2042.09300,6849.0025,1194.8000,26863.0825,7533.0500,1034.2375,28808.6175
1,2025-08-01 11:00:00+00:00,55177.31725,59343.8600,8682.429142,1379.14475,30348.800059,3482.707719,96.3500,0.4600,2038.85600,7461.6800,1301.4750,28281.8100,8082.9900,1302.7025,30079.5675
2,2025-08-01 12:00:00+00:00,53434.16450,57282.3850,9311.592149,1577.60075,30740.298962,3465.497778,146.9600,50.0650,2035.09825,8372.5875,1490.2475,28508.6700,8770.8850,1562.4300,29379.8725
3,2025-08-01 13:00:00+00:00,52269.03625,56441.8225,9216.993201,2200.89650,29765.103199,3509.954172,117.5550,126.9450,2148.24300,9080.5675,1655.8150,26987.6200,9057.3550,1863.7825,28131.0525
4,2025-08-01 14:00:00+00:00,51821.41200,55508.4750,9966.386070,2366.87275,26315.437914,3640.781295,145.1550,54.1050,2127.30875,9397.5975,1812.6225,23324.9675,9316.4875,2215.9575,25338.9150
5,2025-08-01 15:00:00+00:00,52501.40575,54927.5450,9343.561905,2224.42750,19974.666977,3799.991409,328.2775,188.8375,2189.51725,9164.9950,1895.3425,18085.1625,9316.9225,2174.5375,19496.2450


Hourly aggregates quality assured:
- [X] All columns (see "Generation Forecast for Wind & Solar" and "Actual Generation per Production Type" under "Generation" on the [ENTSO-E Transparency Platform](https://transparency.entsoe.eu/generation/actual/perType/generation?appState=%7B%22sa%22%3A%5B%22BZN%7C10Y1001A1001A82H%22%5D%2C%22st%22%3A%22BZN%22%2C%22mm%22%3Atrue%2C%22ma%22%3Afalse%2C%22sp%22%3A%22HALF%22%2C%22dt%22%3A%22TABLE%22%2C%22df%22%3A%5B%222026-03-19%22%2C%222026-03-19%22%5D%2C%22tz%22%3A%22CET%22%7D))

## Netztransparenz (UTC)

In [15]:
show_api_windows('Netztransparenz (UTC)', API_CONFIG['Netztransparenz (UTC)'])


Netztransparenz (UTC)

Window A: 2021-03-01 10:00:00+00:00 -> 2021-03-01 15:00:00+00:00 (timestamp_utc) | rows=6


,timestamp_utc,NRV_balance_qs,NRV_balance_op,NRV_balance,reBAP_shortage_surplus,rz_saldo_mw_qs,rz_saldo_mw_op,rz_saldo_mw,afrr_activated_mw_pos,afrr_activated_mw_neg,mfrr_activated_mw_pos,mfrr_activated_mw_neg,afrr_picasso_mw_pos,afrr_picasso_mw_neg,mfrr_mari_mw_pos,mfrr_mari_mw_neg
0,2021-03-01 10:00:00+00:00,-72.066,-69.56675,-72.066,60.1875,-70.77275,-69.56675,-70.77275,13.875,117.717,0.0,0.0,NaN,NaN,NaN,NaN
1,2021-03-01 11:00:00+00:00,-232.432,-233.46350,-232.432,-7.8675,-232.55950,-233.46350,-232.55950,25.384,214.968,0.0,0.0,NaN,NaN,NaN,NaN
2,2021-03-01 12:00:00+00:00,-14.875,-15.63450,-14.875,65.6075,-14.50500,-15.63450,-14.50500,89.668,162.015,0.0,0.0,NaN,NaN,NaN,NaN
3,2021-03-01 13:00:00+00:00,-155.099,-157.22550,-155.099,33.4725,-155.09925,-157.22550,-155.09925,2.853,178.760,0.0,0.0,NaN,NaN,NaN,NaN
4,2021-03-01 14:00:00+00:00,-423.018,-423.25625,-423.018,18.9425,-422.77450,-423.25625,-422.77450,2.636,302.594,0.0,0.0,NaN,NaN,NaN,NaN
5,2021-03-01 15:00:00+00:00,-205.896,-206.14575,-205.896,-15.7950,-206.28925,-206.14575,-206.28925,52.062,210.484,0.0,0.0,NaN,NaN,NaN,NaN



Window B: 2025-08-01 10:00:00+00:00 -> 2025-08-01 15:00:00+00:00 (timestamp_utc) | rows=6


,timestamp_utc,NRV_balance_qs,NRV_balance_op,NRV_balance,reBAP_shortage_surplus,rz_saldo_mw_qs,rz_saldo_mw_op,rz_saldo_mw,afrr_activated_mw_pos,afrr_activated_mw_neg,mfrr_activated_mw_pos,mfrr_activated_mw_neg,afrr_picasso_mw_pos,afrr_picasso_mw_neg,mfrr_mari_mw_pos,mfrr_mari_mw_neg
0,2025-08-01 10:00:00+00:00,-219.349,-226.1735,-219.349,5.4275,-219.349,-226.1735,-219.349,0.000,171.295,0.0,0.0,1023.476,1049.796,134.982,134.947
1,2025-08-01 11:00:00+00:00,244.598,243.4375,244.598,4.8200,244.598,243.4375,244.598,0.000,69.136,0.0,0.0,1090.731,772.420,170.907,170.902
2,2025-08-01 12:00:00+00:00,-295.153,-286.5740,-295.153,-8.5325,-295.153,-286.5740,-295.153,0.027,66.686,0.0,0.0,582.402,831.123,181.539,181.556
3,2025-08-01 13:00:00+00:00,-968.516,-962.8770,-968.516,-19.6225,-968.516,-962.8770,-968.516,0.094,112.983,0.0,0.5,309.046,1137.905,218.336,217.525
4,2025-08-01 14:00:00+00:00,-153.134,-147.5720,-153.134,9.3050,-153.134,-147.5720,-153.134,18.631,0.929,0.0,9.0,220.973,454.185,247.638,238.957
5,2025-08-01 15:00:00+00:00,515.359,516.4780,515.359,170.7575,515.359,516.4780,515.359,498.478,0.000,0.0,0.0,1007.143,998.851,123.302,123.273


The NRV balance and the sum of the regional values (RZ-saldo) are nearly identical because the NRV balance mathematically aggregates the localized physical imbalances of all four German Transmission System Operators (TSOs) into a single, commercial nationwide deficit or surplus. The minor deviations observed (typically < 10 MW) are merely the result of accounting artifacts, such as virtual metering points and subsequent measurement corrections by the TSOs.

Hourly aggregates quality assured:
- [X] afrr_activated_mw_pos, afrr_activated_mw_neg, mfrr_activated_mw_pos and mfrr_activated_mw_neg (see [Aktivierte Regelleistung](https://www.netztransparenz.de/de-de/Regelenergie/Daten-Regelreserve/Aktivierte-Regelleistung))
- [X] NRV_balance_qs, NRV_balance_op and NRV_balance (see [NRV-Saldo (viertelstündlich)](https://www.netztransparenz.de/de-de/Regelenergie/NRV-und-RZ-Saldo/NRV-Saldo-viertelst%C3%BCndlich))
- [X] rz_saldo_mw_qs, rz_saldo_mw_op and rz_saldo_mw (see [RZ-Saldo](https://www.netztransparenz.de/de-de/Regelenergie/NRV-und-RZ-Saldo/RZ-Saldo))
- [X] reBAP_shortage_surplus (see [reBAP](https://www.netztransparenz.de/de-de/Regelenergie/Ausgleichsenergiepreis/reBAP))
- [X] afrr_picasso_mw_pos and afrr_picasso_mw_neg (see [Optimierungen aus MARI/PICASSO](https://www.netztransparenz.de/de-de/Regelenergie/Daten-Regelreserve/Optimierte-Regelleistung))

## Regelleistung (CET/CEST)

In [16]:
show_api_windows('Regelleistung (CET/CEST)', API_CONFIG['Regelleistung (CET/CEST)'])


Regelleistung (CET/CEST)

Window A: 2021-03-01 10:00:00+01:00 -> 2021-03-01 15:00:00+01:00 (timestamp_cet) | rows=6


,timestamp_cet,afrr_capacity_offered_mw_neg,afrr_capacity_offered_mw_pos,afrr_capacity_price_neg,afrr_capacity_price_pos,afrr_avg_activation_price_neg,afrr_avg_activation_price_pos,afrr_activation_offered_mw_neg,afrr_activation_offered_mw_pos,net_import_export_mw,afrr_bid_avg_activation_price_neg,afrr_bid_avg_activation_price_pos,afrr_bid_vwap_activation_price_neg,afrr_bid_vwap_activation_price_pos,bid_alloc_mw_neg,bid_alloc_mw_pos
0,2021-03-01 10:00:00+01:00,3560.0,4808.0,1.7275,17.1675,-124.73,2554.23,2628.0,2244.0,-20.5,-750.0,9999.99,-124.731286,2554.227812,2077.0,2244.0
1,2021-03-01 11:00:00+01:00,3560.0,4808.0,1.7275,17.1675,-124.73,2554.23,2628.0,2244.0,-20.5,-750.0,9999.99,-124.731286,2554.227812,2077.0,2244.0
2,2021-03-01 12:00:00+01:00,3606.0,4738.0,4.4500,5.7800,-192.70,460.21,2310.0,2292.0,0.0,-2000.0,2000.00,-192.703075,460.207591,2120.0,2237.0
3,2021-03-01 13:00:00+01:00,3606.0,4738.0,4.4500,5.7800,-192.70,460.21,2310.0,2292.0,0.0,-2000.0,2000.00,-192.703075,460.207591,2120.0,2237.0
4,2021-03-01 14:00:00+01:00,3606.0,4738.0,4.4500,5.7800,-192.70,460.21,2310.0,2292.0,0.0,-2000.0,2000.00,-192.703075,460.207591,2120.0,2237.0
5,2021-03-01 15:00:00+01:00,3606.0,4738.0,4.4500,5.7800,-192.70,460.21,2310.0,2292.0,0.0,-2000.0,2000.00,-192.703075,460.207591,2120.0,2237.0



Window B: 2025-08-01 10:00:00+02:00 -> 2025-08-01 15:00:00+02:00 (timestamp_cet) | rows=6


,timestamp_cet,afrr_capacity_offered_mw_neg,afrr_capacity_offered_mw_pos,afrr_capacity_price_neg,afrr_capacity_price_pos,afrr_avg_activation_price_neg,afrr_avg_activation_price_pos,afrr_activation_offered_mw_neg,afrr_activation_offered_mw_pos,net_import_export_mw,afrr_bid_avg_activation_price_neg,afrr_bid_avg_activation_price_pos,afrr_bid_vwap_activation_price_neg,afrr_bid_vwap_activation_price_pos,bid_alloc_mw_neg,bid_alloc_mw_pos
0,2025-08-01 10:00:00+02:00,3805.0,4321.0,9.02,9.53,-1153.4175,1206.8625,2080.00,2269.25,-39.5,-15000.0,15000.0,-1153.563737,1206.834790,8320.0,9077.0
1,2025-08-01 11:00:00+02:00,3805.0,4321.0,9.02,9.53,-1124.6900,1275.9025,2048.50,2288.75,-39.5,-15000.0,15000.0,-1124.363905,1275.104805,8194.0,9155.0
2,2025-08-01 12:00:00+02:00,3925.0,4311.0,35.88,5.67,-1396.2875,1447.4000,1994.50,2166.50,-39.5,-15000.0,15000.0,-1396.153826,1447.310787,7978.0,8666.0
3,2025-08-01 13:00:00+02:00,3925.0,4311.0,35.88,5.67,-1525.1850,1456.4875,1936.75,2204.50,-39.5,-15000.0,15000.0,-1525.292148,1456.463319,7747.0,8818.0
4,2025-08-01 14:00:00+02:00,3925.0,4311.0,35.88,5.67,-1554.6325,1449.2575,1909.00,2219.25,-39.5,-15000.0,15000.0,-1554.643013,1449.189516,7636.0,8877.0
5,2025-08-01 15:00:00+02:00,3925.0,4311.0,35.88,5.67,-1570.6750,1498.2825,1909.50,2213.75,-39.5,-15000.0,15000.0,-1570.680843,1498.407503,7638.0,8855.0


Hourly aggregates quality assured:
- [X] afrr_avg_activation_price_pos, afrr_avg_activation_price_neg, afrr_activation_offered_mw_pos, afrr_activation_offered_mw_neg (see [Regelleistung.net Data Center](https://www.regelleistung.net/apps/datacenter/tendering-files/?productTypes=aFRR&markets=ENERGY&fileTypes=RESULTS&dateRange=2025-08-01%2C2025-08-31))
- [ ] afrr_capacity_offered_mw_neg, afrr_capacity_offered_mw_pos, -- did not match
- [ ] afrr_capacity_price_neg, afrr_capacity_price_pos,
- [ ] net_import_export_mw,


## SMARD (CET/CEST)

In [17]:
show_api_windows('SMARD (CET/CEST)', API_CONFIG['SMARD (CET/CEST)'])


SMARD (CET/CEST)

Window A: 2021-03-01 10:00:00+01:00 -> 2021-03-01 15:00:00+01:00 (timestamp_cet) | rows=6


,timestamp_cet,residual_load_actual,wind_offshore_actual,solar_actual,wind_onshore_forecast,wind_offshore_forecast,solar_forecast,wind_onshore_actual,generation_fossil_brown_coal_mw,generation_fossil_hard_coal_mw,generation_fossil_gas_mw,generation_nuclear_mw,generation_hydro_pumped_storage_mw,da_price_eur,price_intraday_eur,wind_forecast_de,wind_onshore_error,wind_offshore_error,solar_error,system_stress_signal,wind_onshore_capacity,wind_offshore_capacity,solar_capacity,gas_capacity,hard_coal_capacity,lignite_capacity,pumped_storage_capacity
0,2021-03-01 10:00:00+01:00,52570.75,448.25,15557.25,1898.25,415.50,17090.00,1814.75,14449.75,5922.75,11297.25,2034.6875,1154.9375,52.40,52.40,2313.75,83.50,-32.75,1532.75,1649.00,54494.0,NaN,7265112.0,32038.0,NaN,NaN,NaN
1,2021-03-01 11:00:00+01:00,49536.25,394.00,19074.25,1749.50,430.00,21615.25,1746.00,14468.75,5693.75,10836.25,2031.2500,1124.1875,44.48,43.40,2179.50,3.50,36.00,2541.00,2580.50,54494.0,NaN,7265112.0,32038.0,NaN,NaN,NaN
2,2021-03-01 12:00:00+01:00,46873.00,358.00,21723.75,1714.50,460.00,23677.75,1674.25,13887.50,4782.75,10201.50,2027.0000,1109.3750,42.05,40.04,2174.50,40.25,102.00,1954.00,2096.25,54494.0,NaN,7265112.0,32038.0,NaN,NaN,NaN
3,2021-03-01 13:00:00+01:00,44352.75,401.75,22751.25,1819.75,506.25,23374.75,1431.50,13455.00,4145.75,10043.75,2027.8750,1109.8125,40.07,39.87,2326.00,388.25,104.50,623.50,1116.25,54494.0,NaN,7265112.0,32038.0,NaN,NaN,NaN
4,2021-03-01 14:00:00+01:00,44430.25,374.00,20763.50,1964.25,560.25,20342.75,1569.25,13500.00,4137.25,10215.00,2023.6875,1114.1875,40.03,40.03,2524.50,395.00,186.25,-420.75,1002.00,54494.0,NaN,7265112.0,32038.0,NaN,NaN,NaN
5,2021-03-01 15:00:00+01:00,48145.50,322.25,15623.00,2169.00,612.75,14676.00,1759.00,13898.50,4080.25,10759.25,2022.5000,1120.7500,41.75,39.90,2781.75,410.00,290.50,-947.00,1647.50,54494.0,NaN,7265112.0,32038.0,NaN,NaN,NaN



Window B: 2025-08-01 10:00:00+02:00 -> 2025-08-01 15:00:00+02:00 (timestamp_cet) | rows=6


,timestamp_cet,residual_load_actual,wind_offshore_actual,solar_actual,wind_onshore_forecast,wind_offshore_forecast,solar_forecast,wind_onshore_actual,generation_fossil_brown_coal_mw,generation_fossil_hard_coal_mw,generation_fossil_gas_mw,generation_nuclear_mw,generation_hydro_pumped_storage_mw,da_price_eur,price_intraday_eur,wind_forecast_de,wind_onshore_error,wind_offshore_error,solar_error,system_stress_signal,wind_onshore_capacity,wind_offshore_capacity,solar_capacity,gas_capacity,hard_coal_capacity,lignite_capacity,pumped_storage_capacity
0,2025-08-01 10:00:00+02:00,29572.71,974.38,21696.57,6175.50,1206.25,19294.75,6577.58,6013.50,460.75,2670.25,NaN,949.9800,97.95,95.40,7381.75,-402.08,231.87,-2401.82,3035.77,63403.0,9215.0,86952.0,36682.0,15951.0,15178.0,9384.0
1,2025-08-01 11:00:00+02:00,22356.05,978.96,26959.72,6432.75,1171.50,23682.00,7358.38,5611.50,417.50,2419.50,NaN,903.4850,84.10,81.53,7604.25,-925.63,192.54,-3277.72,4395.89,63403.0,9215.0,86952.0,36682.0,15951.0,15178.0,9384.0
2,2025-08-01 12:00:00+02:00,17690.75,1294.96,29888.62,6849.00,1195.00,26863.00,8081.39,4249.25,365.50,2153.25,NaN,885.2775,77.05,64.84,8044.00,-1232.39,-99.96,-3025.62,4357.97,63403.0,9215.0,86952.0,36682.0,15951.0,15178.0,9384.0
3,2025-08-01 13:00:00+02:00,14766.94,1379.14,30348.80,7461.75,1301.25,28281.75,8671.49,3655.25,391.25,1865.25,NaN,870.6750,61.97,51.79,8763.00,-1209.74,-77.89,-2067.05,3354.68,63403.0,9215.0,86952.0,36682.0,15951.0,15178.0,9384.0
4,2025-08-01 14:00:00+02:00,11804.67,1577.60,30740.30,8372.75,1490.00,28508.50,9258.35,3630.50,338.25,1877.50,NaN,866.3750,49.06,41.29,9862.75,-885.60,-87.60,-2231.80,3205.00,63403.0,9215.0,86952.0,36682.0,15951.0,15178.0,9384.0
5,2025-08-01 15:00:00+02:00,11086.04,2200.90,29765.10,9080.50,1656.00,26987.75,9183.07,3641.25,347.25,1912.00,NaN,877.4900,49.10,39.59,10736.50,-102.57,-544.90,-2777.35,3424.82,63403.0,9215.0,86952.0,36682.0,15951.0,15178.0,9384.0


## yfinance (UTC)

In [18]:
show_api_windows('yfinance (UTC)', API_CONFIG['yfinance (UTC)'])


yfinance (UTC)

Window A: 2021-03-01 10:00:00+00:00 -> 2021-03-01 15:00:00+00:00 (timestamp_utc) | rows=6


,timestamp_utc,gas_price_ttf,coal_price_api2,co2_price_eua
0,2021-03-01 10:00:00+00:00,16.08,65.5,58.540001
1,2021-03-01 11:00:00+00:00,16.08,65.5,58.540001
2,2021-03-01 12:00:00+00:00,16.08,65.5,58.540001
3,2021-03-01 13:00:00+00:00,16.08,65.5,58.540001
4,2021-03-01 14:00:00+00:00,16.08,65.5,58.540001
5,2021-03-01 15:00:00+00:00,16.08,65.5,58.540001



Window B: 2025-08-01 10:00:00+00:00 -> 2025-08-01 15:00:00+00:00 (timestamp_utc) | rows=6


,timestamp_utc,gas_price_ttf,coal_price_api2,co2_price_eua
0,2025-08-01 10:00:00+00:00,33.972,103.550003,68.129997
1,2025-08-01 11:00:00+00:00,33.972,103.550003,68.129997
2,2025-08-01 12:00:00+00:00,33.972,103.550003,68.129997
3,2025-08-01 13:00:00+00:00,33.972,103.550003,68.129997
4,2025-08-01 14:00:00+00:00,33.972,103.550003,68.129997
5,2025-08-01 15:00:00+00:00,33.972,103.550003,68.129997
